# ANGSI JR. V9.1 — HPC JupyterLab launcher
Run on the same HPC host as your seismic files.

In [ ]:
import os, sys, urllib.request, subprocess, time
from pathlib import Path
app=Path('/data/epic20/APGCE/apgce_team_13/ANGSI_JR_V9_1_FULL_COVERAGE/angsi_jr_v9.py')
assert app.exists(), f'App missing: {app}'
print('Python:',sys.executable,'App:',app)


## Install requirements if needed

In [ ]:
%pip install -r /data/epic20/APGCE/apgce_team_13/ANGSI_JR_V9_1_FULL_COVERAGE/requirements.txt


## Start Streamlit on remote port 8514

In [ ]:
env=os.environ.copy()
env['ANGSI_DATA_ROOT']='/data/epic20/APGCE/input_data/challenge3'
log=app.parent/'server.log'
with log.open('a') as out:
    proc=subprocess.Popen([sys.executable,'-m','streamlit','run',str(app),
        '--server.address=127.0.0.1','--server.port=8514','--server.headless=true',
        '--server.fileWatcherType=none'],cwd=str(app.parent),stdout=out,
        stderr=subprocess.STDOUT,env=env,start_new_session=True)
print('Server PID:',proc.pid,'Log:',log)


## Server health

In [ ]:
try:
    with urllib.request.urlopen('http://127.0.0.1:8514/_stcore/health',timeout=5) as r:
        print('Server status:',r.status,r.read().decode())
except Exception as e:
    print('Server health failed:',e)
    print(log.read_text(errors='replace')[-4000:] if log.exists() else 'No log')
